In [1]:
"""
Usage:
pip install git+https://github.com/macrocosm-os/prompting.git@feature/organic-task

Reward is generated based on:
    1. Relevance:
        Cosine similarity between the response and the reference embeddings,
        generated by WhereIsAI/UAE-Large-V1 AngleE model.
    2. Rouge:
        Comparison of overlap of n-grams, word sequences, and word pairs between the response and reference.
"""
import torch
from prompting.agent import HumanAgent
from prompting.dendrite import DendriteResponseEvent, SynapseStreamResult
from prompting.organic.organic_task import OrganicTask
from prompting.rewards.pipeline import RewardPipeline
from prompting.rewards.reward import RewardResult
from prompting.protocol import StreamPromptingSynapse

MESSAGES = [
    "What is the capital of Texas",
    "What is the capital of France",
    "When was electricity invented?",
    "How many litres in a kg?",
    "What is the speed of light?",
    "When was the first computer invented?"
]
REFS = [
    "The capital of Texas is Austin.",
    "The capital of France is Paris.",
    (
        "Electricity as a concept has been known for thousands of years, but the modern study of electricity began "
        "in the 17th and 18th centuries with scientists like William Gilbert and Benjamin Franklin. "
        "The invention of the practical use of electricity can be attributed to Thomas Edison and his development of the "
        "electric light bulb in the late 19th century."
    ),
    (
        "The conversion between liters and kilograms depends on the substance. For water, 1 liter of water is "
        "approximately 1 kilogram because the density of water is roughly 1 kg/L."
    ),
    (
        "The speed of light in a vacuum is approximately 299,792,458 meters per second (about 300,000 kilometers "
        "per second or 186,282 miles per second)."
    ),
    (
        "The concept of a programmable computer dates back to Charles Babbage's design of the Analytical Engine "
        "in the 1830s. However, the first general-purpose electronic computer, ENIAC (Electronic Numerical "
        "Integrator and Computer), was completed in 1945."
    )
]

uids = list(range(len(MESSAGES)))

/workspace/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-07-26 01:02:09,169	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
reward_pipeline = RewardPipeline(
    selected_tasks=[OrganicTask.name],
    device="cpu",
    available_tasks={OrganicTask.name: OrganicTask},
)

def generate_reward(msg: str, res: str, ref: str, uid: int) -> float:
    sample = {"roles": ["user"], "messages": [msg]}
    task = OrganicTask(context=sample, reference=ref)

    # Dummy HumanAgent used to reuse existing reward pipeline.
    agent = HumanAgent(
        task=task,
        llm_pipeline=None,
        begin_conversation=True,
        system_prompt="",
    )

    stream_results = [
        SynapseStreamResult(
            accumulated_chunks=[res],
            accumulated_chunks_timings=[0.],  # Dummy value: 0.0 seconds for response.
            tokens_per_chunk=[len(res.split()) * 1.3],  # Approx num of tokens.
            synapse=StreamPromptingSynapse(roles=["user"], messages=[msg], completion=res),
            uid=uid,
        )
    ]
    response_event = DendriteResponseEvent(stream_results=stream_results, uids=torch.tensor([uid]), timeout=15)

    reward_result = RewardResult(
        reward_pipeline,
        agent=agent,
        response_event=response_event,
        device="cpu",
    )
    return reward_result.rewards[0]

In [3]:
import bittensor as bt
import argparse
from prompting.utils.uids import get_random_uids
from prompting.validator import Validator
from prompting.forward import handle_response

WALLET = "opentensor"
HOTKEY = "main"
NETWORK = "finney"
NET_UID = 1

parser = argparse.ArgumentParser()
parser.add_argument("--wallet.name", type=str, default=WALLET)
parser.add_argument("--wallet.hotkey", type=str, default=HOTKEY)
parser.add_argument("--neuron.axon_off", action="store_true", help="Axon off", default=True)
parser.add_argument("--netuid", type=int, default=NET_UID)
parser.add_argument("--subtensor.network", type=str, default=NETWORK)
parser.add_argument("--neuron.model_id", type=str, default="casperhansen/llama-3-8b-instruct-awq")
parser.add_argument("--neuron.llm_max_allowed_memory_in_gb", type=int, default=24)
parser.add_argument("--neuron.sample_size", type=int, default=3)
parser.add_argument("--neuron.disable_set_weights", action="store_true", help="Disables setting weights.", default=True)


config = bt.config(parser=parser)
validator = Validator(config=config)
assert validator.config.neuron.disable_set_weights


async def query_miners(msg: str) -> dict:
    roles = ["user"]
    messages = [msg]

    uids = get_random_uids(validator, k=validator.config.neuron.sample_size, exclude=None).to(validator.device)
    uids_cpu = uids.cpu().tolist()
    print(f"Querying miner UIDs: {uids_cpu}")
    streams_responses = await validator.dendrite.forward(
        axons=[validator.metagraph.axons[uid] for uid in uids_cpu],
        synapse=StreamPromptingSynapse(roles=roles, messages=messages),
        timeout=15,
        deserialize=False,
        streaming=True,
    )
    stream_results_dict = dict(zip(uids_cpu, streams_responses))
    responses = await handle_response(stream_results_dict, validator.llm_pipeline.tokenizer)
    uid_to_response = {}
    for uid, response in zip(uids_cpu, responses):
        uid_to_response[uid] = response.synapse.completion
    return uid_to_response

INFO:bittensor: - Logging path: /root/.bittensor/miners/opentensor/main/netuid1/validator - 
INFO:bittensor: - 
no_prompt: false
wallet:
  name: opentensor
  hotkey: main
  path: ~/.bittensor/wallets/
subtensor:
  network: finney
  chain_endpoint: wss://entrypoint-finney.opentensor.ai:443
  _mock: false
logging:
  debug: false
  trace: false
  record_log: false
  logging_dir: ~/.bittensor/miners
axon:
  port: 8091
  ip: '[::]'
  external_port: null
  external_ip: null
  max_workers: 10
netuid: 1
neuron:
  device: cuda
  gpus: 1
  llm_max_allowed_memory_in_gb: 24
  epoch_length: 100
  events_retention_size: 2 GB
  dont_save_events: false
  log_full: false
  name: validator
  model_id: casperhansen/llama-3-8b-instruct-awq
  tasks:
  - qa
  - date_qa
  - summarization
  - generic
  - math
  - translation
  - sentiment
  task_p:
  - 0.14285714285714285
  - 0.14285714285714285
  - 0.14285714285714285
  - 0.14285714285714285
  - 0.14285714285714285
  - 0.14285714285714285
  - 0.1428571428571

2024-07-26 01:02:26.251 |       INFO       |  - Logging path: /root/.bittensor/miners/opentensor/main/netuid1/validator - 
2024-07-26 01:02:26.268 |       INFO       |  - 
no_prompt: false
wallet:
  name: opentensor
  hotkey: main
  path: ~/.bittensor/wallets/
subtensor:
  network: finney
  chain_endpoint: wss://entrypoint-finney.opentensor.ai:443
  _mock: false
logging:
  debug: false
  trace: false
  record_log: false
  logging_dir: ~/.bittensor/miners
axon:
  port: 8091
  ip: '[::]'
  external_port: null
  external_ip: null
  max_workers: 10
netuid: 1
neuron:
  device: cuda
  gpus: 1
  llm_max_allowed_memory_in_gb: 24
  epoch_length: 100
  events_retention_size: 2 GB
  dont_save_events: false
  log_full: false
  name: validator
  model_id: casperhansen/llama-3-8b-instruct-awq
  tasks:
  - qa
  - date_qa
  - summarization
  - generic
  - math
  - translation
  - sentiment
  task_p:
  - 0.14285714285714285
  - 0.14285714285714285
  - 0.14285714285714285
  - 0.14285714285714285
  - 0.1

INFO:bittensor: - Connected to finney network and wss://entrypoint-finney.opentensor.ai:443. - 
INFO:bittensor: - Wallet: wallet(opentensor, main, ~/.bittensor/wallets/) - 
INFO:bittensor: - Subtensor: subtensor(finney, wss://entrypoint-finney.opentensor.ai:443) - 
INFO:bittensor: - Metagraph: metagraph(netuid:1, n:1024, block:3465955, network:finney) - 
INFO:bittensor: - Running neuron on subnet: 1 with uid 5 using network: wss://entrypoint-finney.opentensor.ai:443 - 
INFO:bittensor: - Dendrite: dendrite(5F4tQyWrhfGVcNhoqeiNsR6KjD4wMZ2kfhLj4oHYuyHbZAc3) - 
INFO:bittensor: - Building validation weights. - 
INFO:bittensor: - Saving validator state. - 
INFO:bittensor: - load_state() - 
INFO:bittensor: - Loading validator state. - 
INFO:bittensor: - Available free memory: 47.29 GB - 
INFO:bittensor: - Total gpu memory 47.62 GB - 
INFO:bittensor: - 51.0% of the GPU memory will be utilized for loading the model to device "cuda". - 
/workspace/venv/lib/python3.10/site-packages/huggingface_hu

WARNING 07-26 01:02:36 config.py:213] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 07-26 01:02:36 llm_engine.py:161] Initializing an LLM engine (v0.4.3) with config: model='casperhansen/llama-3-8b-instruct-awq', speculative_config=None, tokenizer='casperhansen/llama-3-8b-instruct-awq', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, disable_custom_all_reduce=False, quantization=awq, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=casperhansen/llama-3-8b-instruct-awq)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


INFO 07-26 01:02:38 weight_utils.py:207] Using model weights format ['*.safetensors']
INFO 07-26 01:02:40 model_runner.py:146] Loading model weights took 5.3440 GB
INFO 07-26 01:02:42 gpu_executor.py:83] # GPU blocks: 7891, # CPU blocks: 2048
INFO 07-26 01:02:44 model_runner.py:854] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 07-26 01:02:44 model_runner.py:858] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 07-26 01:02:55 model_runner.py:924] Graph capturing finished in 11 secs.


INFO:bittensor: - Supported language pairs: [English -> French, English -> Portuguese, English -> Spanish, English -> Ukranian, French -> English, Portuguese -> English, Portuguese -> Spanish, Spanish -> English, Spanish -> Portuguese, Ukranian -> English] - 
Checking installed packages:   0%|          | 0/10 [00:00<?, ?it/s]INFO:bittensor: - Package from en to fr is already installed, skipping... - 
INFO:bittensor: - Package from en to pt is already installed, skipping... - 
INFO:bittensor: - Package from en to es is already installed, skipping... - 
INFO:bittensor: - Package from en to uk is already installed, skipping... - 
INFO:bittensor: - Package from fr to en is already installed, skipping... - 
INFO:bittensor: - Package from pt to en is already installed, skipping... - 
INFO:bittensor: - Package from pt to es is already installed, skipping... - 
INFO:bittensor: - Package from es to en is already installed, skipping... - 
INFO:bittensor: - Package from es to pt is already instal

In [4]:
# Rewards showcase for responses given by random miners.
for msg, ref in zip(MESSAGES, REFS):
    uid_to_response = await query_miners(msg)
    uid = next(iter(uid_to_response))
    response = uid_to_response[uid]
    reward = generate_reward(msg, response, ref, uid)
    print(f"Message: {msg}\nResponse: {response}\nReference: {ref}\nReward: {reward}\n")
    print("============\n\n")

Querying miner UIDs: [883, 402, 792]


INFO:bittensor: - 🤖 Generating challenge query... - 


Message: What is the capital of Texas
Response: Austin
Reference: The capital of Texas is Austin.
Reward: 0.3374101221561432



Querying miner UIDs: [930, 438, 45]


INFO:bittensor: - 🤖 Generating challenge query... - 


Message: What is the capital of France
Response: The answer to that is: Paris!
Reference: The capital of France is Paris.
Reward: 0.28278905153274536



Querying miner UIDs: [271, 1009, 533]


...

Message: How many litres in a kg?
Response: according to the provided context, 1 kg of a substance is equal to 1,000 millilitres (mL) or 1 litre. Therefore, to find the number of liters in 1 kg of a substance, we can convert 1 kg to liters as 

1 kg = 1,000 mL = 1 L

so, there is **1 litre** in 1 kg of a substance.
Reference: The conversion between liters and kilograms depends on the substance. For water, 1 liter of water is approximately 1 kilogram because the density of water is roughly 1 kg/L.
Reward: 0.3134877681732178



Querying miner UIDs: [819, 867, 317]


INFO:bittensor: - 🤖 Generating challenge query... - 


Message: What is the speed of light?
Response: The speed of light in a vacuum is approximately 299,792,458 meters per second (m/s) or 186,282 miles per second (mi/s). This is a fundamental constant of the universe, denoted by the letter c, and it is the maximum speed at which any object or information can travel.
Reference: The speed of light in a vacuum is approximately 299,792,458 meters per second (about 300,000 kilometers per second or 186,282 miles per second).
Reward: 0.45413538813591003



Querying miner UIDs: [1012, 496, 374]


INFO:bittensor: - 🤖 Generating challenge query... - 


Message: When was the first computer invented?
Response: the invention of the first computer is a matter of debate among historians and computer scientists. There were several machines that could be considered the first computer, depending on how one defines a "computer." Here are a few examples:

1. **Antikythera mechanism** (circa 100 BC): This ancient Greek analog computer was used to calculate astronomical positions and predict eclipses. It was discovered in a shipwreck off the Greek island of Antikythera in 1900. 2. **Abacus** (circa 2500 BC): The abacus is a simple calculating device that has been used for thousands of years. It is a precursor to more complex calculating machines. 3. **Napier's bones** (1617): These were a set of rods with numbers engraved on them that could be used to perform arithmetic operations. They were invented by John Napier, a Scottish mathematician. 4. **Blaise Pascal's calculator** (1642): Pascal's calculator was the first mechanical calculator that co

In [5]:
# Hard-coded responses reward showcase.
RESPONSES = [
    "",  # Empty
    "The capital of France is Paris.",  # Correct
    "Electricity was invented in the 18th century by Thomas Edison.",  # Incorrect
    "There are 2 liters in a kilogram.",  # Incorrect
    "The speed of light is approximately 300,000 kilometers per second.",  # Correct
    "The first computer was invented in 1985."  # Incorrect
]
for msg, ref, res, uid in zip(MESSAGES, REFS, RESPONSES, uids):
    reward = generate_reward(msg, res, ref, uid)
    print(f"Q: {msg}\nA: {res}\nR: {ref}\nReward: {reward}\n======\n")

INFO:bittensor: - 🤖 Generating challenge query... - 


Q: What is the capital of Texas
A: 
R: The capital of Texas is Austin.
Reward: 0.0



INFO:bittensor: - 🤖 Generating challenge query... - 


Q: What is the capital of France
A: The capital of France is Paris.
R: The capital of France is Paris.
Reward: 0.5475575923919678



INFO:bittensor: - 🤖 Generating challenge query... - 


...

Q: How many litres in a kg?
A: There are 2 liters in a kilogram.
R: The conversion between liters and kilograms depends on the substance. For water, 1 liter of water is approximately 1 kilogram because the density of water is roughly 1 kg/L.
Reward: 0.2674546539783478



INFO:bittensor: - 🤖 Generating challenge query... - 


Q: What is the speed of light?
A: The speed of light is approximately 300,000 kilometers per second.
R: The speed of light in a vacuum is approximately 299,792,458 meters per second (about 300,000 kilometers per second or 186,282 miles per second).
Reward: 0.442534863948822



INFO:bittensor: - 🤖 Generating challenge query... - 


Q: When was the first computer invented?
A: The first computer was invented in 1985.
R: The concept of a programmable computer dates back to Charles Babbage's design of the Analytical Engine in the 1830s. However, the first general-purpose electronic computer, ENIAC (Electronic Numerical Integrator and Computer), was completed in 1945.
Reward: 0.265046089887619

